<a href="https://colab.research.google.com/github/rsahoo44691/MSAI-631-traditional-chatbot-/blob/main/chatbot_azure_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet azure-ai-textanalytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 14.6 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports and credential loading
import random
from datetime import datetime

from google.colab import userdata
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

ENDPOINT_URI = userdata.get('MicrosoftAIServiceEndpoint')
API_KEY = userdata.get('MicrosoftAPIKey')

assert ENDPOINT_URI, 'Add MicrosoftAIServiceEndpoint to Colab Secrets and enable notebook access.'
assert API_KEY, 'Add MicrosoftAPIKey to Colab Secrets and enable notebook access.'

print('Endpoint loaded:', ENDPOINT_URI.split('//')[1].split('.')[0] + '.cognitiveservices.azure.com/')
print('API key loaded:', '*' * 8 + API_KEY[-4:])

Endpoint loaded: msai631-language.cognitiveservices.azure.com/
API key loaded: ********hky2


In [4]:
# Cell 3 — Bot identity, capabilities, and helpers
BOT_NAME = 'SentiBot'
BOT_VERSION = '1.0'

CAPABILITIES = [
    "Tell you the sentiment of any message (positive, neutral, or negative).",
    "Show the three confidence scores returned by Azure AI Language.",
    "Respond differently depending on how positive or negative you sound.",
    "List my capabilities when you type 'help' or 'what can you do'.",
    "Handle empty input and gibberish without crashing.",
    "Say goodbye when you type 'bye', 'quit', or 'exit'.",
]

POSITIVE_REPLIES = [
    "Glad to hear that!",
    "That sounds great.",
    "Love the positive energy.",
]

NEGATIVE_REPLIES = [
    "I'm sorry to hear that.",
    "That sounds tough — thanks for sharing.",
    "Sounds frustrating.",
]

NEUTRAL_REPLIES = [
    "Got it.",
    "Understood.",
    "Thanks for letting me know.",
]

MIXED_REPLIES = [
    "I'm picking up mixed signals there.",
    "Sounds like a bit of both.",
]

EXIT_WORDS = {'bye', 'quit', 'exit', 'goodbye'}
HELP_WORDS = {'help', 'what can you do', 'capabilities', 'commands'}

def list_capabilities() -> str:
    bullets = '\n- '.join(CAPABILITIES)
    return f"Here is what I can do:\n- {bullets}"

In [5]:
# Cell 4 — Build the Azure AI Language client
text_client = TextAnalyticsClient(
    endpoint=ENDPOINT_URI,
    credential=AzureKeyCredential(API_KEY),
)

# Quick smoke test
smoke = text_client.analyze_sentiment(documents=['I love this assignment.'])[0]
print('Smoke test sentiment:', smoke.sentiment)
print('Confidence:', smoke.confidence_scores)

Smoke test sentiment: positive
Confidence: {'positive': 1.0, 'neutral': 0.0, 'negative': 0.0}


In [6]:
# Cell 5 — get_response(): the single entry point
def analyze(text: str):
    """Return (label, confidence_scores) from Azure or (None, None) on error."""
    try:
        result = text_client.analyze_sentiment(documents=[text])[0]
        if result.is_error:
            return None, None
        return result.sentiment, result.confidence_scores
    except Exception as ex:
        print(f"[Azure error: {type(ex).__name__}: {ex}]")
        return None, None

def format_scores(scores) -> str:
    return (
        f"(positive {scores.positive:.2f}, "
        f"neutral {scores.neutral:.2f}, "
        f"negative {scores.negative:.2f})"
    )

def pick_reply(label: str) -> str:
    if label == 'positive':
        return random.choice(POSITIVE_REPLIES)
    if label == 'negative':
        return random.choice(NEGATIVE_REPLIES)
    if label == 'mixed':
        return random.choice(MIXED_REPLIES)
    return random.choice(NEUTRAL_REPLIES)

def get_response(user_input: str) -> str:
    if not user_input or not user_input.strip():
        return "It looks like you did not type anything. Try saying 'hello' or 'help'."

    cleaned = user_input.strip()
    lowered = cleaned.lower()

    if lowered in HELP_WORDS:
        return list_capabilities()

    if lowered in EXIT_WORDS:
        return 'Goodbye!'

    # Everything else goes to Azure for sentiment analysis.
    label, scores = analyze(cleaned)
    if label is None:
        return "I could not reach the sentiment service. Please try again."

    reply = pick_reply(label)
    return f"{reply}\nSentiment: {label.upper()}  {format_scores(scores)}"

In [7]:
# Cell 6 — Interactive chat loop
def run_chat():
    print(f"{BOT_NAME} v{BOT_VERSION}: Hi! Type a message and I will read its sentiment. Type 'help' for what I can do, or 'bye' to quit.\n")
    while True:
        user = input('You: ')
        reply = get_response(user)
        print(f"{BOT_NAME}: {reply}\n")
        if user.strip().lower() in EXIT_WORDS:
            break

run_chat()

SentiBot v1.0: Hi! Type a message and I will read its sentiment. Type 'help' for what I can do, or 'bye' to quit.

You: Hello
SentiBot: Got it.
Sentiment: NEUTRAL  (positive 0.05, neutral 0.95, negative 0.01)

You: I love this assignment
SentiBot: Love the positive energy.
Sentiment: POSITIVE  (positive 1.00, neutral 0.00, negative 0.00)

You: this is the worst day ever
SentiBot: I'm sorry to hear that.
Sentiment: NEGATIVE  (positive 0.00, neutral 0.00, negative 1.00)

You: the weather is okay
SentiBot: Thanks for letting me know.
Sentiment: NEUTRAL  (positive 0.26, neutral 0.73, negative 0.01)

You: it was great food but terrible service
SentiBot: I'm sorry to hear that.
Sentiment: NEGATIVE  (positive 0.01, neutral 0.00, negative 0.98)

You: help
SentiBot: Here is what I can do:
- Tell you the sentiment of any message (positive, neutral, or negative).
- Show the three confidence scores returned by Azure AI Language.
- Respond differently depending on how positive or negative you sound